#### Similar Applications: Choosing the Retrieval Method

In [2]:
# Libraries; the outcome table (with its date split) and the deployed risk model for comparison

import time

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score
from transformers import AutoModel, AutoTokenizer

from planscope.outcome_data import OUT_PATH
from planscope.outcome_model import OUT_PATH as RISK_PATH
from planscope.risk import RiskModel

DEVICE = 'cuda'

'cuda'

In [3]:
# Past applications to search (train + val years) and 3,000 later test-year applications as queries: no peeking at the future

df = pd.read_parquet(OUT_PATH)
index = df[df['split'].isin(['train', 'val'])].reset_index(drop=True)
queries = df[df['split'] == 'test'].sample(3000, random_state=0).reset_index(drop=True)
print(f'{len(index):,} past applications to search, {len(queries):,} test-year queries')

570,453 past applications to search, 3,000 test-year queries


In [4]:
# Two small embedding models; each turns a description into 384 numbers that capture its meaning

MODELS = {'minilm': ('sentence-transformers/all-MiniLM-L6-v2', 'mean'), 'bge-small': ('BAAI/bge-small-en-v1.5', 'cls')}
encoders = {}
for name, (model_name, pooling) in MODELS.items():
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    encoders[name] = (AutoTokenizer.from_pretrained(model_name), model.half() if DEVICE == 'cuda' else model, pooling)


@torch.inference_mode()
def embed(name, texts, batch_size=512):
    tokenizer, model, pooling = encoders[name]
    out = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True, max_length=128, return_tensors='pt').to(DEVICE)
        hidden = model(**batch).last_hidden_state
        mask = batch['attention_mask'].unsqueeze(-1)
        vectors = hidden[:, 0] if pooling == 'cls' else (hidden * mask).sum(1) / mask.sum(1)
        out.append(torch.nn.functional.normalize(vectors.float(), dim=1).cpu().numpy())
    return np.vstack(out)


vectors = {}
for name in MODELS:
    start = time.perf_counter()
    vectors[name] = embed(name, index['text'].tolist()), embed(name, queries['text'].tolist())
    print(f'{name}: {time.perf_counter() - start:.0f} s')

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

C:\Users\samad\anaconda3\envs\planscope\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\samad\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\samad\anaconda3\envs\planscope\Lib\site-packages\huggingface_hub\file_download.py

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

... (9 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

minilm: 164 s
bge-small: 233 s


In [5]:
# Find each query's 10 most similar past applications in its own borough, by each method

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_index, X_query = vectorizer.fit_transform(index['text']), vectorizer.transform(queries['text'])


def search(similarity, k=10):
    found = np.zeros((len(queries), k), dtype=int)
    for borough, q in queries.groupby('lpa_name').groups.items():
        i = np.flatnonzero(index['lpa_name'] == borough)
        found[q] = i[np.argsort(-similarity(np.asarray(q), i), axis=1)[:, :k]]
    return found


found = {'tf-idf': search(lambda q, i: (X_query[q] @ X_index[i].T).toarray())}
for name, (E, Q) in vectors.items():
    found[name] = search(lambda q, i: Q[q] @ E[i].T)

In [10]:
# Find each query's 10 most similar past applications by each method; `by` sets what must match (borough, or borough and type)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_index, X_query = vectorizer.fit_transform(index['text']), vectorizer.transform(queries['text'])


def search(similarity, by=('lpa_name',), k=10):
    found = [None] * len(queries)
    for key, q in queries.groupby(list(by)).indices.items():
        key = key if isinstance(key, tuple) else (key,)
        i = np.flatnonzero(np.logical_and.reduce([index[col] == value for col, value in zip(by, key)]))
        top = i[np.argsort(-similarity(np.asarray(q), i), axis=1)[:, :k]]
        for row, neighbours in zip(q, top):
            found[row] = neighbours
    return found


def tfidf(q, i):
    return (X_query[q] @ X_index[i].T).toarray()


def dense(name):
    return lambda q, i: vectors[name][1][q] @ vectors[name][0][i].T


def hybrid(name, pool=50, weight=0.5):
    # Embeddings shortlist the 50 closest; the final order mixes meaning (embedding) and exact wording (TF-IDF)
    def similarity(q, i):
        d, lexical = dense(name)(q, i), tfidf(q, i)
        keep = np.full_like(d, -np.inf)
        np.put_along_axis(keep, np.argsort(-d, axis=1)[:, :pool], 0, axis=1)
        return weight * d + (1 - weight) * lexical + keep
    return similarity


found = {'tf-idf': search(tfidf)}
for name in MODELS:
    found[name] = search(dense(name))

In [11]:
# Are the neighbours good evidence? Same type as the query, and does their refusal rate predict the real decision?

def evidence(found):
    knn_risk = np.array([index['refused'].to_numpy()[f].mean() for f in found])
    same_type = np.mean([(index['category'].to_numpy()[f] == c).mean() for f, c in zip(found, queries['category'])])
    return {'same_type@10': same_type, 'roc_auc': roc_auc_score(queries['refused'], knn_risk),
            'pr_auc': average_precision_score(queries['refused'], knn_risk)}


risk = RiskModel(RISK_PATH)
model_risk = [risk.predict(t, b, c)['risk'] for t, b, c in queries[['text', 'lpa_name', 'category']].to_numpy()]
table = pd.DataFrame({name: evidence(f) for name, f in found.items()}).T
table.loc['outcome model', ['roc_auc', 'pr_auc']] = [roc_auc_score(queries['refused'], model_risk), average_precision_score(queries['refused'], model_risk)]
table.round(3)

,same_type@10,roc_auc,pr_auc
tf-idf,0.706,0.700,0.322
minilm,0.767,0.703,0.316
bge-small,0.766,0.718,0.336
outcome model,NaN,0.759,0.411


In [12]:
# int8 storage: a scoring function using compressed vectors, and how much of the top-10 it keeps

def int8(name):
    E, Q = vectors[name]
    scale = np.abs(E).max(axis=1, keepdims=True) / 127
    E8 = np.round(E / scale).astype(np.int8)
    return lambda q, i: Q[q] @ (E8[i] * scale[i]).T


{name: round(np.mean([len(set(a) & set(b)) / 10 for a, b in zip(search(int8(name)), found[name])]), 3) for name in MODELS}

{'minilm': np.float64(0.937), 'bge-small': np.float64(0.913)}

In [13]:
# Everyday wording, and direction ("house into flats" vs "flats into house"): top 3 in Camden by each method

pd.set_option('display.max_colwidth', 90)
camden = np.flatnonzero(index['lpa_name'] == 'Camden')


def top3(text):
    scores = {'tf-idf': (vectorizer.transform([text]) @ X_index[camden].T).toarray()[0]}
    for name in MODELS:
        d = vectors[name][0][camden] @ embed(name, [text])[0]
        scores[name] = d
        shortlist = np.isin(np.arange(len(d)), np.argsort(-d)[:50])
        scores[f'{name} + tf-idf'] = np.where(shortlist, 0.5 * d + 0.5 * scores['tf-idf'], -np.inf)
    return pd.DataFrame({name: index['text'].to_numpy()[camden[np.argsort(-s)[:3]]] for name, s in scores.items()})


for text in ['big extension on the back of my house with a terrace on the roof', 'cut down the tree in my front garden',
             'convert my house into flats', 'convert two flats back into one house']:
    print(text)
    display(top3(text))

big extension on the back of my house with a terrace on the roof


,tf-idf,minilm,minilm + tf-idf,bge-small,bge-small + tf-idf
0,Erection of a dormer extension on the rear roofslope and installation of 3 rooflights ...,Roof extension with front roof terrace.,Erection of roof extension with terrace to front,Rear roof extension with roof terrace.,Erection of an extension on roof terrace.
1,Erection of roof extension with terrace to front,Rear roof extension with roof terrace.,Erection of an extension on roof terrace.,Erection of an extension on roof terrace.,Erection of roof extension with terrace to front
2,Erection of mansard roof extension with terrace,Erection of roof extension with terrace to front,Roof extension with front roof terrace.,Roof extension with front roof terrace.,Erection of a mansard roof extension with terrace.


cut down the tree in my front garden


,tf-idf,minilm,minilm + tf-idf,bge-small,bge-small + tf-idf
0,Replacement of front garden outbuilding.,"Alterations to front boundary wall, steps, side gate and landscaping of front garden i...","Alterations to front boundary wall, steps, side gate and landscaping of front garden i...",Alterations to front garden landscaping.,Alterations to front garden landscaping.
1,Alterations to front garden landscaping.,Erection of a timber bin enclosure to front garden.,Alterations to front garden landscaping.,Alterations to hard and soft landscaping to front garden (retrospective),Replacement of front garden outbuilding.
2,Erection of side extension (on existing single storey wing on south elevation) and rem...,Demolition of existing timber outbuilding in rear garden.,Erection of a timber bin enclosure to front garden.,Replacement of front garden outbuilding.,Alterations to hard and soft landscaping to front garden (retrospective)


convert my house into flats


,tf-idf,minilm,minilm + tf-idf,bge-small,bge-small + tf-idf
0,"Proposal to convert existing dwelling house into 2 x self- contained flats, installati...",Conversion of two flats into one residential unit,Conversion of 2 flats into a single dwelling.,Conversion of 1 x 3-bed and 1 x 2-bed flats into 1 x dwellinghouse,Conversion of 2 flats into a single dwelling.
1,"Conversion of 3-bed flat into 3 flats (Class C3) including 2 storey rear extension, si...",Conversion of 2 flats into a single dwelling.,Conversion of two flats into one residential unit,Conversion of 1 x 3-bed and 1 x 1-bed flats into 1 x dwellinghouse,Conversion of two flats into one dwellinghouse.
2,"Conversion of 1 x 5 bed dwellinghouse into 2 flats and replacement of front, side and ...",Conversion of two flats into one dwellinghouse.,"Proposal to convert existing dwelling house into 2 x self- contained flats, installati...",Conversion of two flats into one residential unit,Conversion of two flats into one residential unit


convert two flats back into one house


,tf-idf,minilm,minilm + tf-idf,bge-small,bge-small + tf-idf
0,Conversion of 2 flats into one house (Class C3).,Conversion of two flats into one residential unit,Conversion of 2 flats into one house (Class C3).,Conversion of two flats into one dwellinghouse.,Conversion of 2 flats into one house (Class C3).
1,"Conversion of two flats back into one single dwelling house, replacement of front and ...",Conversion of two flats into one dwellinghouse.,"Conversion of two flats back into one single dwelling house, replacement of front and ...",Conversion of two flats into one residential unit,Conversion of two flats into one dwellinghouse.
2,Conversion of two flats into one dwellinghouse.,Conversion of 2 flats into a single dwelling.,Conversion of two flats into one dwellinghouse.,Conversion of 2 flats into a single dwelling.,Conversion of two flats into one residential unit


In [16]:
# Final check: TF-IDF within the same borough and application type (the production choice)
same_type = search(tfidf, ('lpa_name', 'category'))
pd.Series(evidence(same_type)).round(3)

same_type@10    1.000
roc_auc         0.721
pr_auc          0.346
dtype: float64

## Findings and decision

**Question:** for a new description, which past decided applications should we show as "similar", and how should we find them?

**Setup:** 570,453 decided applications from the train and val years as the index; 3,000 test-year applications as queries. "Evidence" = how well the refusal share among a query's 10 nearest neighbours predicts its actual outcome (ROC-AUC, PR-AUC).

| Method | Same type in top 10 | ROC-AUC | PR-AUC |
|---|---|---|---|
| TF-IDF, borough only | 0.706 | 0.700 | 0.322 |
| MiniLM, borough only | 0.767 | 0.703 | 0.316 |
| BGE-small, borough only | 0.766 | 0.718 | 0.336 |
| **TF-IDF, borough + type** | **1.000** | **0.721** | **0.346** |
| Outcome model (for reference) | – | 0.759 | 0.411 |

**What we learned**
- Embeddings found more same-type neighbours, but filtering by type gives every method 100%, so that advantage disappears.
- Embeddings are blind to direction: "convert my house into flats" returned flats converted into houses for both models, and a 50/50 hybrid with TF-IDF did not fix it. TF-IDF's word pairs ("house into", "into flats") get the direction right.
- int8 quantisation kept 91–94% of the top-10 neighbours, but was not needed once embeddings were dropped.
- The neighbours (0.721) are weaker predictors than the outcome model (0.759), so the model stays the risk score; the neighbours are evidence a user can read.
- Caveat: queries were filtered on their true type. In production the type is sometimes predicted (macro-F1 0.888), so live results will be slightly worse.

**Decision:** TF-IDF cosine similarity within the same borough and type, reusing the outcome model's vocabulary (`planscope.similar`). No extra model; numpy-only search in about 11 ms; index about 390 MB. The full same-type comparison of every method (the original cell 11) was dropped: the decision rule set beforehand (switch only if an embedding method won by 0.02 ROC-AUC or more) could not be met, given the direction failure and a 0.018 gap without the type filter.